
<div style="text-align: right;">
  <a href="../../Files/review_resume_proxy.zip"
     download="review_resume_proxy.zip"
     class="md-button md-button--primary">
    ⬇️ Download Notebook
  </a>


## Install and import utility.ipynb

In [1]:
%run ./utility.ipynb

In [ ]:
%pip install -e ./utility.ipynb
%pip install import_ipynb

In [3]:
import import_ipynb
import utility

## Authenticate and get token

In [ ]:
token = utility.get_oauth_token()

## Build Prompt

In [5]:
prompt = """You are an expert career advisor and AI job-matching assistant.
                        I will provide a resume as a pdf. First, extract the text from the resume.
                        Then analyze it to determine the most relevant job roles the candidate is suited for.
                        For each suggested job role, provide:
                        1. Fit Score (0–100) with reasoning
                        2. Matched Skills (from the resume)
                        3. Skill Gaps (skills missing or weak compared to role requirements)
                        4. Targeted Improvement Guidance (practical, specific suggestions to close the skill gaps)
                        Output should be structured in JSON with the following format:
                        {
                        "candidate_summary": "short summary of candidate’s profile",
                        "recommended_roles": [
                        {
                            "role": "job title",
                            "fit_score": number,
                            "matched_skills": ["skill1", "skill2", ...],
                            "skill_gaps": ["missing_skill1", "missing_skill2", ...],
                            "improvement_guidance": "clear, actionable recommendations"
                        }
                        ]
                        }
                        The resume is provided as a pdf """

## Build request body

In [6]:
base64encoded = utility.base64_encode_file(pdf_path='D:/Users/572981/Downloads/example_usecase_refactored/resume_example.pdf')
inferenceConfig = {
            "maxTokens": 512,
            "temperature": 0.5,
            "topP": 0.9
        }
body = {
        "inferenceConfig": inferenceConfig,
        "messages": [
            {   "role": "user",
                "content": [
                    {"text": prompt},
                    {"document": {
                            "format": "pdf",
                            "name":'resume',
                            "source": {
                                "bytes": base64encoded}}
                    }]
            }]
        }
request_body={
    "methodName" :  "converse",
    "parameters" : {
        "body" : body
    }
}

## Invoke Modelops

In [7]:
modelId = "anthropic.claude-3-5-sonnet-20241022-v2:0"
jobId = utility.invoke_llm_proxy(model_id=modelId, token=token, request_body=request_body)

## Poll job status via Get Job by job ID until succeeded/failed and Get Data is Successfull

In [ ]:
if utility.wait_for_job_completion(jobId, token):
    modelops_resp = utility.get_response(jobId, token)
    response={}
    response["body"] = json.dumps(modelops_resp["output"]['message']['content'][0]['text'])
else:
    response = {}

print(response)